<a href="https://colab.research.google.com/github/zen-zap/pheno_crop/blob/main/Pheno_Crop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pheno Crop

Detection of crop growth stage from satellite imagery

In [1]:
# GEE setup

import ee

ee.Authenticate()

ee.Initialize(project='pheno-crop-489815')
print("Earth Engine Initialized Successfully!")

Earth Engine Initialized Successfully!


In [21]:
from google.colab import userdata
PROJECT_ID = userdata.get('PROJECT_ID')

print(PROJECT_ID)

pheno-crop-489815


## Scripts Setup

In [24]:
%%writefile gee_index_fetcher.py

"""
GEE – Sentinel-2 Index Fetcher + Image Downloader
==================================================
Pulls vegetation indices AND downloads satellite images directly from
Google Earth Engine for any lat/lon location over a date range.

Image types available:
  rgb         -> True colour (B4, B3, B2)
  falsecolor  -> False colour NIR composite (B8, B4, B3)
  ndvi        -> NDVI heatmap (colourised, red to green)
  geotiff     -> Raw multi-band GeoTIFF (all S2 bands)

Export destinations:
  local       -> PNG/TIF saved to --image-dir on your machine
  drive       -> Exported to your Google Drive (async GEE task)
  both        -> Local + Drive
"""

import argparse
import os
import sys
import requests
from datetime import datetime, timedelta

# ==============================================================================
# GLOBAL CONFIGURATION
# ==============================================================================
DRIVE_EXPORT_FOLDER = "sentinel_2"  # Folder created at the root of Google Drive

# ──────────────────────────────────────────────────────────────────────────────
# INDEX REGISTRY
# ──────────────────────────────────────────────────────────────────────────────
INDEX_REGISTRY = {
    "NDVI":  ("(NIR - RED) / (NIR + RED)",                           ["B8", "B4"]),
    "EVI":   ("2.5 * (NIR - RED) / (NIR + 6*RED - 7.5*BLUE + 1)",   ["B8", "B4", "B2"]),
    "EVI2":  ("2.5 * (NIR - RED) / (NIR + 2.4*RED + 1)",            ["B8", "B4"]),
    "SAVI":  ("1.5 * (NIR - RED) / (NIR + RED + 0.5)",              ["B8", "B4"]),
    "MSAVI": ("(2*NIR + 1 - sqrt((2*NIR+1)**2 - 8*(NIR-RED))) / 2", ["B8", "B4"]),
    "GNDVI": ("(NIR - GREEN) / (NIR + GREEN)",                       ["B8", "B3"]),
    "NDRE":  ("(NIR - REDEDGE) / (NIR + REDEDGE)",                  ["B8", "B5"]),
    "NDWI":  ("(GREEN - NIR) / (GREEN + NIR)",                       ["B3", "B8"]),
    "NDMI":  ("(NIR - SWIR1) / (NIR + SWIR1)",                      ["B8", "B11"]),
    "NBR":   ("(NIR - SWIR2) / (NIR + SWIR2)",                      ["B8", "B12"]),
    "ARVI":  ("(NIR - (2*RED - BLUE)) / (NIR + (2*RED - BLUE))",    ["B8", "B4", "B2"]),
}
ALL_INDEX_NAMES = list(INDEX_REGISTRY.keys())

IMAGE_TYPES    = ["rgb", "falsecolor", "ndvi", "geotiff"]
EXPORT_TARGETS = ["local", "drive", "both"]

# NDVI colour ramp: red (bare/stressed) -> yellow -> green (healthy)
NDVI_PALETTE = [
    "#d73027", "#f46d43", "#fdae61", "#fee08b",
    "#ffffbf", "#d9ef8b", "#a6d96a", "#66bd63",
    "#1a9850", "#006837",
]


# ──────────────────────────────────────────────────────────────────────────────
# 1. ARGUMENT PARSING
# ──────────────────────────────────────────────────────────────────────────────

def parse_args():
    parser = argparse.ArgumentParser(
        description="Fetch Sentinel-2 indices + download images from GEE.",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=__doc__,
    )
    parser.add_argument("--lat",   type=float, required=True, help="Latitude  (e.g. 25.5620).")
    parser.add_argument("--lon",   type=float, required=True, help="Longitude (e.g. 84.8720).")
    parser.add_argument("--start", required=True, help="Start date YYYY-MM-DD.")
    parser.add_argument("--end",   required=True, help="End date   YYYY-MM-DD.")
    parser.add_argument("--plot-id", required=True, type=str, help="The ID of the plot (e.g., 2).")
    parser.add_argument("--gap",   type=int, default=None,
                        help="Gap in days for composite windows. Omit = every scene.")
    parser.add_argument("--indices", nargs="+", default=ALL_INDEX_NAMES,
                        choices=ALL_INDEX_NAMES, metavar="INDEX",
                        help=f"Indices to compute. Default: all.")
    parser.add_argument("--buffer", type=int,   default=500, help="AOI buffer in metres (default 500).")
    parser.add_argument("--cloud",  type=float, default=20,  help="Max cloud %% (default 20).")
    parser.add_argument("--scale",  type=int,   default=10,  help="Pixel resolution metres (default 10).")
    parser.add_argument("--image-types", nargs="+", default=[], choices=IMAGE_TYPES, metavar="TYPE",
                        help=f"Image types to download: {IMAGE_TYPES}. Default: none (indices only).")
    parser.add_argument("--export", default="local", choices=EXPORT_TARGETS,
                        help="Export destination: local | drive | both (default: local).")
    parser.add_argument("--image-dir", default="./gee_images",
                        help="Local folder to save images (default: ./gee_images).")
    parser.add_argument("--output", default=None, help="CSV output path.")
    return parser.parse_args()


# ──────────────────────────────────────────────────────────────────────────────
# 2. GEE INIT
# ──────────────────────────────────────────────────────────────────────────────

def init_gee():
    PROJECT_ID = os.environ.get('PROJECT_ID')
    try:
        import ee
    except ImportError:
        print("ERROR: earthengine-api not installed.\n    Run: pip install earthengine-api")
        sys.exit(1)
    try:
        ee.Initialize(project=PROJECT_ID)
        print("OK   Google Earth Engine initialised.")
    except Exception:
        print("WARN Not authenticated. Running: earthengine authenticate ...")
        ee.Authenticate()
        ee.Initialize(project=PROJECT_ID)
    return ee


# ──────────────────────────────────────────────────────────────────────────────
# 3. INDEX COMPUTATION
# ──────────────────────────────────────────────────────────────────────────────

def add_indices(image, ee, index_names):
    s2 = image.divide(10000)
    aliases = {
        "BLUE":    s2.select("B2"),
        "GREEN":   s2.select("B3"),
        "RED":     s2.select("B4"),
        "REDEDGE": s2.select("B5"),
        "NIR":     s2.select("B8"),
        "SWIR1":   s2.select("B11"),
        "SWIR2":   s2.select("B12"),
    }
    for name in index_names:
        expr, _ = INDEX_REGISTRY[name]
        try:
            image = image.addBands(image.expression(expr, aliases).rename(name))
        except Exception as ex:
            print(f"  WARN Could not compute {name}: {ex}")
    return image


# ──────────────────────────────────────────────────────────────────────────────
# 4. VALUE EXTRACTION
# ──────────────────────────────────────────────────────────────────────────────

def extract_values(image, aoi, index_names, scale, ee):
    result = image.select(index_names).reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=scale, maxPixels=1e9
    )
    return result.getInfo()


# ──────────────────────────────────────────────────────────────────────────────
# 5. LOCAL IMAGE SAVE
# ──────────────────────────────────────────────────────────────────────────────

def _download_url(url, dest_path):
    r = requests.get(url, timeout=180)
    r.raise_for_status()
    with open(dest_path, "wb") as f:
        f.write(r.content)


def save_image_locally(image, aoi, label, image_types, image_dir, scale, ee):
    os.makedirs(image_dir, exist_ok=True)
    safe = label.replace(" ", "_")

    for itype in image_types:

        if itype == "rgb":
            vis = {"bands": ["B4", "B3", "B2"], "min": 0.0, "max": 0.3, "gamma": 1.4}
            src = image.divide(10000)
            fname = os.path.join(image_dir, f"{safe}_rgb.png")

        elif itype == "falsecolor":
            vis = {"bands": ["B8", "B4", "B3"], "min": 0.0, "max": 0.5, "gamma": 1.4}
            src = image.divide(10000)
            fname = os.path.join(image_dir, f"{safe}_falsecolor.png")

        elif itype == "ndvi":
            src = image.normalizedDifference(["B8", "B4"]).rename("NDVI")
            vis = {"bands": ["NDVI"], "min": -0.2, "max": 0.8, "palette": NDVI_PALETTE}
            fname = os.path.join(image_dir, f"{safe}_ndvi.png")

        elif itype == "geotiff":
            url = image.getDownloadURL({
                "bands": ["B2", "B3", "B4", "B5", "B8", "B11", "B12"],
                "region": aoi,
                "scale": scale,
                "format": "GEO_TIFF",
            })
            fname = os.path.join(image_dir, f"{safe}_raw.tif")
            try:
                _download_url(url, fname)
                print(f"    SAVE GeoTIFF -> {fname}")
            except Exception as ex:
                print(f"    WARN GeoTIFF download failed: {ex}")
            continue

        else:
            continue

        try:
            url = src.getThumbURL({"region": aoi, "format": "png", **vis})
            _download_url(url, fname)
            print(f"    SAVE {itype.upper()} PNG -> {fname}")
        except Exception as ex:
            print(f"    WARN {itype} save failed: {ex}")


# ──────────────────────────────────────────────────────────────────────────────
# 6. GOOGLE DRIVE EXPORT
# ──────────────────────────────────────────────────────────────────────────────

def export_to_drive(image, aoi, label, image_types, drive_folder, scale, ee):
    safe = label.replace(" ", "_").replace("-", "")

    for itype in image_types:
        if itype == "rgb":
            export_img = image.divide(10000).select(["B4", "B3", "B2"])
            desc = f"{safe}_rgb"
        elif itype == "falsecolor":
            export_img = image.divide(10000).select(["B8", "B4", "B3"])
            desc = f"{safe}_falsecolor"
        elif itype == "ndvi":
            export_img = image.normalizedDifference(["B8", "B4"]).rename("NDVI")
            desc = f"{safe}_ndvi"
        elif itype == "geotiff":
            export_img = image.select(["B2", "B3", "B4", "B5", "B8", "B11", "B12"])
            desc = f"{safe}_raw"
        else:
            continue

        task = ee.batch.Export.image.toDrive(
            image=export_img,
            description=desc,
            folder=drive_folder,
            region=aoi,
            scale=scale,
            maxPixels=1e9,
            fileFormat="GeoTIFF",
        )
        task.start()
        print(f"    DRIVE Export submitted: {desc}")


# ──────────────────────────────────────────────────────────────────────────────
# 7. SCENE FETCH LOOPS
# ──────────────────────────────────────────────────────────────────────────────

def fetch_all_scenes(collection, aoi, args, ee):
    import pandas as pd
    records  = []
    img_list = collection.toList(collection.size())
    n        = img_list.size().getInfo()
    print(f"  Found {n} scenes. Extracting ...\n")

    for i in range(n):
        img       = ee.Image(img_list.get(i))
        date_str  = img.date().format("YYYY-MM-dd").getInfo()
        cloud_pct = img.get("CLOUDY_PIXEL_PERCENTAGE").getInfo()
        img       = add_indices(img, ee, args.indices)
        vals      = extract_values(img, aoi, args.indices, args.scale, ee)

        row = {"date": date_str, "cloud_pct": round(cloud_pct, 2)}
        row.update(vals)
        records.append(row)
        print(f"  [{i+1}/{n}] {date_str}  cloud={cloud_pct:.1f}%  NDVI={vals.get('NDVI', 'N/A')}")

        if args.image_types:
            if args.export in ("local", "both"):
                save_image_locally(img, aoi, date_str, args.image_types, args.image_dir, args.scale, ee)
            if args.export in ("drive", "both"):
                file_prefix = f"plot_{args.plot_id}_{date_str}"
                export_to_drive(img, aoi, file_prefix, args.image_types, DRIVE_EXPORT_FOLDER, args.scale, ee)

    return pd.DataFrame(records)


def fetch_gap_composites(collection, aoi, args, ee):
    import pandas as pd
    records  = []
    current  = datetime.strptime(args.start, "%Y-%m-%d")
    end_dt   = datetime.strptime(args.end,   "%Y-%m-%d")
    win_n    = 0

    while current <= end_dt:
        window_end = min(current + timedelta(days=args.gap - 1), end_dt)
        s     = current.strftime("%Y-%m-%d")
        e     = window_end.strftime("%Y-%m-%d")
        label = f"{s}_to_{e}"
        win_n += 1

        sub   = collection.filterDate(s, e)
        count = sub.size().getInfo()

        if count == 0:
            print(f"  [{win_n}] {s} -> {e}  (no scenes)")
        else:
            composite = sub.median()
            composite = add_indices(composite, ee, args.indices)
            vals      = extract_values(composite, aoi, args.indices, args.scale, ee)

            row = {"period_start": s, "period_end": e, "scene_count": count}
            row.update(vals)
            records.append(row)
            print(f"  [{win_n}] {s} -> {e}  scenes={count}  NDVI={vals.get('NDVI', 'N/A')}")

            if args.image_types:
                if args.export in ("local", "both"):
                    save_image_locally(composite, aoi, label, args.image_types, args.image_dir, args.scale, ee)
                if args.export in ("drive", "both"):
                  file_prefix = f"plot_{args.plot_id}_{label}"
                  export_to_drive(composite, aoi, file_prefix, args.image_types, DRIVE_EXPORT_FOLDER, args.scale, ee)

        current = window_end + timedelta(days=1)

    return pd.DataFrame(records)


# ──────────────────────────────────────────────────────────────────────────────
# 8. MAIN
# ──────────────────────────────────────────────────────────────────────────────

def main():
    args = parse_args()

    print("\nGEE - Sentinel-2 Index Fetcher + Image Downloader")
    print("-" * 58)
    print(f"  Location    : lat={args.lat}, lon={args.lon}")
    print(f"  Date range  : {args.start}  ->  {args.end}")
    print(f"  Gap         : {args.gap if args.gap else 'None (every scene)'} days")
    print(f"  Indices     : {args.indices}")
    print(f"  Image types : {args.image_types if args.image_types else 'None (indices only)'}")
    if args.image_types:
        print(f"  Export to   : {args.export}")
        if args.export in ("local", "both"):
            print(f"  Image dir   : {args.image_dir}")
        if args.export in ("drive", "both"):
            print(f"  Drive folder: {DRIVE_EXPORT_FOLDER}")

    print(f"  Buffer      : {args.buffer} m  |  Cloud: {args.cloud}%  |  Scale: {args.scale} m")
    print()

    ee = init_gee()

    # AOI
    point = ee.Geometry.Point([args.lon, args.lat])
    aoi   = point.buffer(args.buffer)

    # Sentinel-2 SR collection
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(args.start, args.end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", args.cloud))
        .sort("system:time_start")
    )

    total = collection.size().getInfo()
    if total == 0:
        print("ERROR: No scenes found. Try relaxing --cloud or widening the date range.")
        sys.exit(1)

    print(f"  Matching scenes in GEE: {total}\n")

    # Fetch
    if args.gap:
        df = fetch_gap_composites(collection, aoi, args, ee)
    else:
        df = fetch_all_scenes(collection, aoi, args, ee)

    # Display
    import pandas as pd
    pd.set_option("display.max_columns", 20)
    pd.set_option("display.width", 130)
    pd.set_option("display.float_format", "{:.4f}".format)

    print(f"\n{'='*58}")
    print(f"  RESULTS  -  {len(df)} rows x {len(df.columns)} columns")
    print(f"{'='*58}")
    print(df.to_string(index=False))

    # Save CSV
    if args.output:
        df.to_csv(args.output, index=False)
        print(f"\nCSV saved -> {args.output}")

    # Summary
    if args.image_types and args.export in ("local", "both"):
        print(f"\nImages saved -> {os.path.abspath(args.image_dir)}/")
    if args.image_types and args.export in ("drive", "both"):
        print(f"\nDrive exports submitted -> folder: '{DRIVE_EXPORT_FOLDER}' (at Drive root)")
        print("Monitor at: https://code.earthengine.google.com/tasks")

    print("\nDone.\n")


if __name__ == "__main__":
    main()

Overwriting gee_index_fetcher.py


In [ ]:
# @title
# %%writefile coordinate_scripts.py

# // Define all plot coordinates
# var plots = [
#   {id: 2, coords: [[84.832155,25.575205], [84.832196,25.575386], [84.832388,25.575348], [84.832341,25.575163]]},
#   {id: 3, coords: [[84.832352,25.57516], [84.832388,25.575348], [84.832591,25.575296], [84.832546,25.575125]]},
#   {id: 4, coords: [[84.832338,25.575025], [84.832505,25.575767], [84.833002,25.575677], [84.83283,25.574947]]},
#   {id: 5, coords: [[84.831688,25.575456], [84.831763,25.575837], [84.83216,25.575807], [84.832082,25.575386]]},
#   {id: 6, coords: [[84.830517,25.575633], [84.830593,25.576054], [84.831004,25.576054], [84.830938,25.575589]]},
#   {id: 7, coords: [[84.830586,25.575238], [84.830647,25.575743], [84.831116,25.575711], [84.831068,25.575185]]},
#   {id: 8, coords: [[84.83028,25.57506], [84.830365,25.575592], [84.831262,25.575476], [84.831166,25.57498]]},
#   {id: 9, coords: [[84.829494,25.575376], [84.829625,25.575976], [84.830161,25.575933], [84.830038,25.575356]]},
#   {id: 10, coords: [[84.829187,25.577356], [84.829364,25.5785], [84.830522,25.578425], [84.830309,25.577263]]},
#   {id: 11, coords: [[84.832613,25.585387], [84.832973,25.585387], [84.832962,25.585235], [84.8326,25.58524]]},
#   {id: 12, coords: [[84.831812,25.584481], [84.832254,25.584427], [84.83223,25.584206], [84.831742,25.584266]]},
#   {id: 13, coords: [[84.832605,25.585396], [84.832984,25.585383], [84.832955,25.585233], [84.832597,25.585236]]},
#   {id: 14, coords: [[84.832831,25.585087], [84.833051,25.585058], [84.833037,25.584829], [84.832828,25.584838]]},
#   {id: 15, coords: [[84.832555,25.584388], [84.832775,25.584364], [84.832756,25.584137], [84.832522,25.584154]]},
#   {id: 16, coords: [[84.828025,25.577031], [84.828160,25.578086], [84.828879,25.577993], [84.828606,25.576985]]},
#   {id: 17, coords: [[84.827460,25.582535], [84.827548,25.582999], [84.828402,25.582956], [84.828409,25.582431]]},
#   {id: 18, coords: [[84.827297,25.582089], [84.827460,25.582535], [84.828409,25.582431], [84.827921,25.582089]]},
#   {id: 19, coords: [[84.829154,25.582608], [84.829174,25.583115], [84.830021,25.582999], [84.829730,25.582480]]},
#   {id: 20, coords: [[84.828646,25.581911], [84.828930,25.582406], [84.829601,25.582382], [84.829283,25.581911]]},
#   {id: 21, coords: [[84.827419,25.581013], [84.827460,25.580469], [84.828077,25.580377], [84.828049,25.580848]]},
#   {id: 22, coords: [[84.826660,25.581972], [84.826735,25.582443], [84.827236,25.582498], [84.827060,25.581905]]},
#   {id: 23, coords: [[84.830105,25.583848], [84.830244,25.584144], [84.830824,25.584047], [84.830631,25.583761]]},
#   {id: 24, coords: [[84.830438,25.584231], [84.830513,25.584458], [84.830743,25.584439], [84.830668,25.584216]]},
#   {id: 25, coords: [[84.828724,25.584739], [84.828739,25.585135], [84.829369,25.585122], [84.829373,25.584709]]},
#   {id: 26, coords: [[84.829640,25.585011], [84.829688,25.585423], [84.830271,25.585504], [84.830218,25.584982]]},
#   {id: 27, coords: [[84.829703,25.583166], [84.829771,25.583573], [84.830432,25.583500], [84.830148,25.583074]]},
#   {id: 28, coords: [[84.828283,25.583345], [84.828330,25.583664], [84.828552,25.583633], [84.828560,25.583287]]},
#   {id: 29, coords: [[84.826598,25.583671], [84.826658,25.583902], [84.827045,25.583844], [84.826956,25.583594]]},
#   {id: 30, coords: [[84.841213,25.576807], [84.841459,25.577653], [84.842125,25.577479], [84.841932,25.576601]]},
#   {id: 31, coords: [[84.840968,25.576198], [84.841213,25.576807], [84.841932,25.576601], [84.841643,25.575929]]},
#   {id: 32, coords: [[84.840565,25.575012], [84.840968,25.576198], [84.841643,25.575929], [84.841213,25.574806]]},
#   {id: 33, coords: [[84.841213,25.574806], [84.841529,25.575565], [84.842151,25.575296], [84.841888,25.574696]]},
#   {id: 34, coords: [[84.841888,25.574696], [84.842151,25.575296], [84.842835,25.575138], [84.842660,25.574474]]},
#   {id: 35, coords: [[84.840655,25.571628], [84.840596,25.572227], [84.841459,25.572088], [84.841326,25.571495]]},
#   {id: 36, coords: [[84.824167,25.571072], [84.824416,25.571996], [84.825023,25.571933], [84.824705,25.570928]]},
#   {id: 37, coords: [[84.825023,25.571126], [84.825302,25.571996], [84.825978,25.571915], [84.825739,25.571108]]},
#   {id: 38, coords: [[84.824989,25.569785], [84.825108,25.570507], [84.825590,25.570354], [84.825323,25.569611]]},
#   {id: 39, coords: [[84.825323,25.569611], [84.825590,25.570354], [84.826124,25.570300], [84.826020,25.569752]]},
#   {id: 40, coords: [[84.830575,25.568693], [84.830851,25.569684], [84.831544,25.569559], [84.831356,25.568640]]},
#   {id: 41, coords: [[84.831356,25.568640], [84.831544,25.569559], [84.832335,25.569434], [84.832206,25.568524]]},
#   {id: 42, coords: [[84.832206,25.568524], [84.832335,25.569434], [84.833314,25.569166], [84.833195,25.568381]]},
#   {id: 43, coords: [[84.831514,25.567596], [84.831514,25.568390], [84.832266,25.568292], [84.832206,25.567489]]},
#   {id: 44, coords: [[84.828508,25.569077], [84.828705,25.569826], [84.829625,25.569862], [84.829467,25.568863]]},
#   {id: 45, coords: [[84.829467,25.568863], [84.829625,25.569862], [84.830683,25.569684], [84.830426,25.568792]]},
#   {id: 46, coords: [[84.824690,25.567079], [84.824987,25.567908], [84.825966,25.567792], [84.825758,25.566918]]},
#   {id: 47, coords: [[84.826826,25.552557], [84.827028,25.554131], [84.828412,25.553988], [84.828152,25.552414]]},
#   {id: 48, coords: [[84.830098,25.552310], [84.830545,25.553819], [84.831900,25.553910], [84.831482,25.552258]]},
#   {id: 49, coords: [[84.823626,25.555796], [84.823669,25.556888], [84.825356,25.556862], [84.825226,25.555666]]},
#   {id: 50, coords: [[84.828945,25.550086], [84.829089,25.551348], [84.830675,25.551192], [84.830401,25.550073]]},
#   {id: 51, coords: [[84.811690,25.551127], [84.811575,25.552245], [84.813060,25.551946], [84.813031,25.551127]]},
#   {id: 52, coords: [[84.820047,25.545685], [84.820742,25.547701], [84.822386,25.547116], [84.822062,25.545413]]},
#   {id: 53, coords: [[84.811888,25.546818], [84.812281,25.547991], [84.813099,25.547716], [84.812795,25.546709]]}
# ];

# // Color palette for different plots
# var colors = ['FF0000', '00FF00', '0000FF', 'FFFF00', 'FF00FF', '00FFFF',
#               'FFA500', '800080', 'FFC0CB', 'A52A2A', '808080', '000080'];

# // Create feature collection from all plots
# var features = [];
# plots.forEach(function(plot) {
#   var polygon = ee.Geometry.Polygon([plot.coords]);
#   var feature = ee.Feature(polygon, {plot_id: plot.id});
#   features.push(feature);
# });

# var allPlots = ee.FeatureCollection(features);

# // Calculate the center of all plots for initial map view
# var bounds = allPlots.geometry().bounds();
# Map.centerObject(bounds, 12);

# // Add all plots to the map with labels
# plots.forEach(function(plot, index) {
#   var polygon = ee.Geometry.Polygon([plot.coords]);
#   var color = colors[index % colors.length];

#   Map.addLayer(polygon, {color: color}, 'Plot ' + plot.id, false);

#   // Add label at centroid
#   var centroid = polygon.centroid();
#   Map.addLayer(centroid, {color: 'FFFFFF'}, 'Label ' + plot.id, false);
# });

# // Add all plots as one layer with different colors
# Map.addLayer(allPlots.style({
#   color: 'blue',
#   fillColor: '00000000',
#   width: 2
# }), {}, 'All Plots', true);

# // Print summary information
# print('Total number of plots:', plots.length);
# print('Plot IDs:', plots.map(function(p) { return p.id; }));
# print('All plots feature collection:', allPlots);

# // Optional: Add a base layer (Sentinel-2 image)
# var sentinel2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
#   .filterBounds(bounds)
#   .filterDate('2024-01-01', '2025-01-01')
#   .sort('CLOUDY_PIXEL_PERCENTAGE')
#   .first();

# Map.addLayer(sentinel2, {
#   bands: ['B4', 'B3', 'B2'],
#   min: 0,
#   max: 3000
# }, 'Sentinel-2 RGB', false);

# // Create a legend
# var legend = ui.Panel({
#   style: {
#     position: 'bottom-left',
#     padding: '8px 15px'
#   }
# });

# var legendTitle = ui.Label({
#   value: 'Plot Legend',
#   style: {
#     fontWeight: 'bold',
#     fontSize: '16px',
#     margin: '0 0 4px 0',
#     padding: '0'
#   }
# });

# legend.add(legendTitle);
# legend.add(ui.Label('Total Plots: ' + plots.length));
# legend.add(ui.Label('Click layers to toggle visibility'));

# Map.add(legend);

Writing coordinate_scripts.py


## Plot Coordinates

In [11]:
# @title
import os
import subprocess
from google.colab import drive

drive.mount('/content/drive')

plots = [
  {"id": 2, "coords": [[84.832155,25.575205], [84.832196,25.575386], [84.832388,25.575348], [84.832341,25.575163]]},
  {"id": 3, "coords": [[84.832352,25.57516], [84.832388,25.575348], [84.832591,25.575296], [84.832546,25.575125]]},
  {"id": 4, "coords": [[84.832338,25.575025], [84.832505,25.575767], [84.833002,25.575677], [84.83283,25.574947]]},
  {"id": 5, "coords": [[84.831688,25.575456], [84.831763,25.575837], [84.83216,25.575807], [84.832082,25.575386]]},
  {"id": 6, "coords": [[84.830517,25.575633], [84.830593,25.576054], [84.831004,25.576054], [84.830938,25.575589]]},
  {"id": 7, "coords": [[84.830586,25.575238], [84.830647,25.575743], [84.831116,25.575711], [84.831068,25.575185]]},
  {"id": 8, "coords": [[84.83028,25.57506], [84.830365,25.575592], [84.831262,25.575476], [84.831166,25.57498]]},
  {"id": 9, "coords": [[84.829494,25.575376], [84.829625,25.575976], [84.830161,25.575933], [84.830038,25.575356]]},
  {"id": 10, "coords": [[84.829187,25.577356], [84.829364,25.5785], [84.830522,25.578425], [84.830309,25.577263]]},
  {"id": 11, "coords": [[84.832613,25.585387], [84.832973,25.585387], [84.832962,25.585235], [84.8326,25.58524]]},
  {"id": 12, "coords": [[84.831812,25.584481], [84.832254,25.584427], [84.83223,25.584206], [84.831742,25.584266]]},
  {"id": 13, "coords": [[84.832605,25.585396], [84.832984,25.585383], [84.832955,25.585233], [84.832597,25.585236]]},
  {"id": 14, "coords": [[84.832831,25.585087], [84.833051,25.585058], [84.833037,25.584829], [84.832828,25.584838]]},
  {"id": 15, "coords": [[84.832555,25.584388], [84.832775,25.584364], [84.832756,25.584137], [84.832522,25.584154]]},
  {"id": 16, "coords": [[84.828025,25.577031], [84.828160,25.578086], [84.828879,25.577993], [84.828606,25.576985]]},
  {"id": 17, "coords": [[84.827460,25.582535], [84.827548,25.582999], [84.828402,25.582956], [84.828409,25.582431]]},
  {"id": 18, "coords": [[84.827297,25.582089], [84.827460,25.582535], [84.828409,25.582431], [84.827921,25.582089]]},
  {"id": 19, "coords": [[84.829154,25.582608], [84.829174,25.583115], [84.830021,25.582999], [84.829730,25.582480]]},
  {"id": 20, "coords": [[84.828646,25.581911], [84.828930,25.582406], [84.829601,25.582382], [84.829283,25.581911]]},
  {"id": 21, "coords": [[84.827419,25.581013], [84.827460,25.580469], [84.828077,25.580377], [84.828049,25.580848]]},
  {"id": 22, "coords": [[84.826660,25.581972], [84.826735,25.582443], [84.827236,25.582498], [84.827060,25.581905]]},
  {"id": 23, "coords": [[84.830105,25.583848], [84.830244,25.584144], [84.830824,25.584047], [84.830631,25.583761]]},
  {"id": 24, "coords": [[84.830438,25.584231], [84.830513,25.584458], [84.830743,25.584439], [84.830668,25.584216]]},
  {"id": 25, "coords": [[84.828724,25.584739], [84.828739,25.585135], [84.829369,25.585122], [84.829373,25.584709]]},
  {"id": 26, "coords": [[84.829640,25.585011], [84.829688,25.585423], [84.830271,25.585504], [84.830218,25.584982]]},
  {"id": 27, "coords": [[84.829703,25.583166], [84.829771,25.583573], [84.830432,25.583500], [84.830148,25.583074]]},
  {"id": 28, "coords": [[84.828283,25.583345], [84.828330,25.583664], [84.828552,25.583633], [84.828560,25.583287]]},
  {"id": 29, "coords": [[84.826598,25.583671], [84.826658,25.583902], [84.827045,25.583844], [84.826956,25.583594]]},
  {"id": 30, "coords": [[84.841213,25.576807], [84.841459,25.577653], [84.842125,25.577479], [84.841932,25.576601]]},
  {"id": 31, "coords": [[84.840968,25.576198], [84.841213,25.576807], [84.841932,25.576601], [84.841643,25.575929]]},
  {"id": 32, "coords": [[84.840565,25.575012], [84.840968,25.576198], [84.841643,25.575929], [84.841213,25.574806]]},
  {"id": 33, "coords": [[84.841213,25.574806], [84.841529,25.575565], [84.842151,25.575296], [84.841888,25.574696]]},
  {"id": 34, "coords": [[84.841888,25.574696], [84.842151,25.575296], [84.842835,25.575138], [84.842660,25.574474]]},
  {"id": 35, "coords": [[84.840655,25.571628], [84.840596,25.572227], [84.841459,25.572088], [84.841326,25.571495]]},
  {"id": 36, "coords": [[84.824167,25.571072], [84.824416,25.571996], [84.825023,25.571933], [84.824705,25.570928]]},
  {"id": 37, "coords": [[84.825023,25.571126], [84.825302,25.571996], [84.825978,25.571915], [84.825739,25.571108]]},
  {"id": 38, "coords": [[84.824989,25.569785], [84.825108,25.570507], [84.825590,25.570354], [84.825323,25.569611]]},
  {"id": 39, "coords": [[84.825323,25.569611], [84.825590,25.570354], [84.826124,25.570300], [84.826020,25.569752]]},
  {"id": 40, "coords": [[84.830575,25.568693], [84.830851,25.569684], [84.831544,25.569559], [84.831356,25.568640]]},
  {"id": 41, "coords": [[84.831356,25.568640], [84.831544,25.569559], [84.832335,25.569434], [84.832206,25.568524]]},
  {"id": 42, "coords": [[84.832206,25.568524], [84.832335,25.569434], [84.833314,25.569166], [84.833195,25.568381]]},
  {"id": 43, "coords": [[84.831514,25.567596], [84.831514,25.568390], [84.832266,25.568292], [84.832206,25.567489]]},
  {"id": 44, "coords": [[84.828508,25.569077], [84.828705,25.569826], [84.829625,25.569862], [84.829467,25.568863]]},
  {"id": 45, "coords": [[84.829467,25.568863], [84.829625,25.569862], [84.830683,25.569684], [84.830426,25.568792]]},
  {"id": 46, "coords": [[84.824690,25.567079], [84.824987,25.567908], [84.825966,25.567792], [84.825758,25.566918]]},
  {"id": 47, "coords": [[84.826826,25.552557], [84.827028,25.554131], [84.828412,25.553988], [84.828152,25.552414]]},
  {"id": 48, "coords": [[84.830098,25.552310], [84.830545,25.553819], [84.831900,25.553910], [84.831482,25.552258]]},
  {"id": 49, "coords": [[84.823626,25.555796], [84.823669,25.556888], [84.825356,25.556862], [84.825226,25.555666]]},
  {"id": 50, "coords": [[84.828945,25.550086], [84.829089,25.551348], [84.830675,25.551192], [84.830401,25.550073]]},
  {"id": 51, "coords": [[84.811690,25.551127], [84.811575,25.552245], [84.813060,25.551946], [84.813031,25.551127]]},
  {"id": 52, "coords": [[84.820047,25.545685], [84.820742,25.547701], [84.822386,25.547116], [84.822062,25.545413]]},
  {"id": 53, "coords": [[84.811888,25.546818], [84.812281,25.547991], [84.813099,25.547716], [84.812795,25.546709]]}
]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Sentinel-2 Fetching Script

In [28]:
base_dir_s2 = '/content/drive/MyDrive/pheno_crop/sentinel_2'
os.makedirs(base_dir_s2, exist_ok=True)

import os, subprocess
PROJECT_ID = userdata.get('PROJECT_ID')
my_env = os.environ.copy()
my_env['PROJECT_ID'] = PROJECT_ID

print(f"Starting batch job for {len(plots)} plots...")

# TODO: Make parameter changes according to season and get plots for all ids

for plot in plots[:1]:
    plot_id = plot['id']
    coords = plot['coords']

    # Calculate Centroid (average lat and lon)
    avg_lon = sum(c[0] for c in coords) / len(coords)
    avg_lat = sum(c[1] for c in coords) / len(coords)

    print(f"Queueing Plot {plot_id}...")

    csv_path = f"{base_dir_s2}/plot_{plot_id}_indices.csv"

    # Command array (Using a very tight 20m buffer so we don't grab neighboring fields)
    cmd = [
        "python", "gee_index_fetcher.py",
        "--lat", str(avg_lat),
        "--lon", str(avg_lon),
        "--start", "2023-09-15",
        "--end", "2023-10-01",
        "--buffer", "20",
        "--plot-id", str(plot_id),
        "--indices", "NDVI", "NDWI", "NDRE", "EVI", "SAVI",
        "--image-types", "geotiff",
        "--export", "drive",
        "--output", csv_path
    ]

    result = subprocess.run(cmd, capture_output=True, text=True, env=my_env)

    if result.returncode == 0:
        print(result.stdout)
    else:
        print(f"❌ Error on Plot {plot_id}:")
        print(result.stderr)

print("All tasks successfully submitted to Google Earth Engine!")
print(f"Check your Google Drive. CSVs are saved to: {base_dir_s2}")
print("GeoTIFF images will appear in Drive in the background as Google's servers finish processing.")

Starting batch job for 52 plots...
Queueing Plot 2...

GEE - Sentinel-2 Index Fetcher + Image Downloader
----------------------------------------------------------
  Location    : lat=25.5752755, lon=84.83227
  Date range  : 2023-09-15  ->  2023-10-01
  Gap         : None (every scene) days
  Indices     : ['NDVI', 'NDWI', 'NDRE', 'EVI', 'SAVI']
  Image types : ['geotiff']
  Export to   : drive
  Drive folder: sentinel_2
  Buffer      : 20 m  |  Cloud: 20%  |  Scale: 10 m

OK   Google Earth Engine initialised.
  Matching scenes in GEE: 2

  Found 2 scenes. Extracting ...

  [1/2] 2023-09-29  cloud=18.3%  NDVI=0.4121679034723098
    DRIVE Export submitted: plot_2_20230929_raw
  [2/2] 2023-09-29  cloud=18.8%  NDVI=0.41103266072199074
    DRIVE Export submitted: plot_2_20230929_raw

  RESULTS  -  2 rows x 7 columns
      date  cloud_pct    EVI   NDRE   NDVI    NDWI   SAVI
2023-09-29    18.3500 0.5485 0.2724 0.4122 -0.3634 0.3213
2023-09-29    18.8100 0.5490 0.2729 0.4110 -0.3620 0.3206

C

## Sentinel-1 Fetching Script

parameters are hardcoded -- so not exactly a script

In [29]:
import ee
import pandas as pd
import os

base_dir_s1 = '/content/drive/MyDrive/pheno_crop/sentinel_1'
os.makedirs(base_dir_s1, exist_ok=True)

# testing with 2 plots
temp_plots = [
  {"id": 2, "coords": [[84.832155,25.575205], [84.832196,25.575386], [84.832388,25.575348], [84.832341,25.575163]]},
  {"id": 3, "coords": [[84.832352,25.57516], [84.832388,25.575348], [84.832591,25.575296], [84.832546,25.575125]]}
]

START_DATE = '2023-05-01'
END_DATE = '2023-10-31'

print(f"📡 Starting Sentinel-1 SAR Extraction for {len(plots)} plots...")


for plot in temp_plots:
    plot_id = plot['id']
    print(f"Processing Plot {plot_id}...")

    # Create the exact polygon from the coordinates
    roi = ee.Geometry.Polygon(plot['coords'])

    # Load Sentinel-1 GRD (Ground Range Detected) Collection
    # Filter for standard agricultural settings (Interferometric Wide swath, ascending/descending)
    s1_collection = (ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(roi)
        .filterDate(START_DATE, END_DATE)
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
        .filter(ee.Filter.eq('instrumentMode', 'IW')))

    # Function to extract mean VV, VH, and calculate Ratio for the plot
    def extract_sar_values(image):
        date = image.date().format('YYYY-MM-dd')

        # Calculate VH/VV Ratio (often used in phenology)
        vv = image.select('VV')
        vh = image.select('VH')
        ratio = vh.subtract(vv).rename('VH_VV_Ratio') # In dB, subtraction is equivalent to division

        img_with_ratio = image.addBands(ratio)

        # Get mean values within the plot polygon
        stats = img_with_ratio.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=roi,
            scale=10, # Sentinel-1 resolution is roughly 10m
            maxPixels=1e9
        )

        return ee.Feature(None, {
            'date': date,
            'VV': stats.get('VV'),
            'VH': stats.get('VH'),
            'VH_VV_Ratio': stats.get('VH_VV_Ratio'),
            'orbit': image.get('orbitProperties_pass') # Keep track of ascending vs descending
        })

    # Map the extraction function over the collection
    extracted_features = s1_collection.map(extract_sar_values)

    # Download the results to Colab memory
    results_list = extracted_features.getInfo()['features']

    # Clean up the output into a Pandas DataFrame
    data = []
    for feat in results_list:
        props = feat['properties']
        if props.get('VV') is not None: # Ensure data isn't null
            data.append({
                'date': props['date'],
                'VV': props['VV'],
                'VH': props['VH'],
                'VH_VV_Ratio': props['VH_VV_Ratio'],
                'orbit': props['orbit']
            })

    df = pd.DataFrame(data)

    # Sort chronologically
    if not df.empty:
        df = df.sort_values('date').reset_index(drop=True)

        csv_path = f"{base_dir_s1}/plot_{plot_id}_sar.csv"
        df.to_csv(csv_path, index=False)
        print(f"Saved {len(df)} SAR records to {csv_path}")
    else:
        print(f"No SAR data found for Plot {plot_id}")

print("SAR Extraction Complete!")

📡 Starting Sentinel-1 SAR Extraction for 52 plots...
Processing Plot 2...
Saved 45 SAR records to /content/drive/MyDrive/pheno_crop/sentinel_1/plot_2_sar.csv
Processing Plot 3...
Saved 45 SAR records to /content/drive/MyDrive/pheno_crop/sentinel_1/plot_3_sar.csv
SAR Extraction Complete!
